In [1]:
import wrds
import pandas as pd
import numpy as np
from typing import Tuple, List, Dict
from pathlib import Path

def g_cmp(d: wrds.Connection, g: str) -> pd.DataFrame:
    """Retrieves global fundamental data from Compustat Global using GVKEY.

    Output schema is normalized to use 'ni' as the net income column name
    (aliased from Compustat Global's native 'nicon') so downstream merges
    and feature engineering can be written once regardless of whether data
    came from the global or North America file.

    Note on datafmt: Compustat Global stores filings under several format
    codes. Issuers with long histories and restatements (BHP among them)
    appear under 'HIST_STD' — the historically-preserved standardized
    format — rather than the plain 'STD' used for more recent entries.
    If you run this loader against a different issuer and it returns zero
    rows, diagnose with:
        SELECT DISTINCT indfmt, datafmt, popsrc, consol, COUNT(*)
        FROM comp.g_funda WHERE gvkey = %(g)s GROUP BY 1,2,3,4

    Args:
        d: Active WRDS connection.
        g: Compustat GVKEY (zero-padded 6-character string).

    Returns:
        DataFrame sorted by datadate. Columns: datadate, curcd, at, lt,
        ni (aliased from nicon), revt. Roughly one row per fiscal year.
    """
    q = """SELECT datadate, curcd, at, lt, nicon AS ni, revt
           FROM comp.g_funda
           WHERE gvkey = %(g)s
             AND indfmt = 'INDL' AND datafmt = 'HIST_STD'
             AND popsrc = 'I' AND consol = 'C'
           ORDER BY datadate ASC"""
    return d.raw_sql(q, params={'g': g}, date_cols=['datadate'])

def g_ibs_int(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves international consensus EPS estimates from IBES summary file.

    Returns FY1 and FY2 annual EPS consensus from ibes.statsum_epsint (the
    split-adjusted international summary file). The fpi column distinguishes
    forecast horizons: '1'-'5' are annual (FY1 through FY5), 'A'-'D' are
    interim half-year estimates, '0' is long-term growth. We keep only the
    annual horizons because interims carry a different fiscal period
    convention that would break time-series merges downstream.

    Note: fpi is stored as a fixed-width character column, so values may
    carry trailing whitespace (e.g. '1 ' rather than '1'). TRIM() is
    required to match reliably — naive equality on '1' silently returns
    zero rows. Similarly, measure is trimmed for defensive consistency.

    Args:
        d: Active WRDS connection.
        t: International IBES ticker (e.g. '@BHP' for BHP Group Ltd).

    Returns:
        DataFrame sorted by statpers. Columns: statpers, fpedats, curcode,
        meanest, medest, numest, stdev, highest, lowest, fpi. Two rows per
        statpers (one for fpi='1', one for fpi='2').
    """
    q = """SELECT statpers, fpedats, curcode, meanest, medest, numest,
                  stdev, highest, lowest, TRIM(fpi) AS fpi
           FROM ibes.statsum_epsint
           WHERE ticker = %(t)s
             AND TRIM(measure) = 'EPS'
             AND TRIM(fpi) IN ('1', '2')
           ORDER BY statpers ASC"""
    return d.raw_sql(q, params={'t': t}, date_cols=['statpers', 'fpedats'])

def g_crs(d: wrds.Connection, p: int, s: str, e: str) -> pd.DataFrame:
    """Retrieves daily stock data from CRSP Version 2 using PERMNO."""
    q = f"SELECT dlycaldt, dlyret, dlyvol FROM crsp.dsf_v2 WHERE permno = {p} AND dlycaldt >= '{s}' AND dlycaldt <= '{e}' ORDER BY dlycaldt ASC"
    return d.raw_sql(q, date_cols=['dlycaldt'])

def g_evt(d: wrds.Connection, c: str) -> pd.DataFrame:
    """Retrieves key developments from Capital IQ using CompanyID."""
    q = f"SELECT a.announcedate, a.keydeveventtypeid, b.headline FROM ciq.wrds_keydev a JOIN ciq.ciqkeydev b ON a.keydevid = b.keydevid WHERE a.companyid = {c} ORDER BY a.announcedate ASC"
    return d.raw_sql(q, date_cols=['announcedate'])

def g_ff(d: wrds.Connection, s: str, e: str) -> pd.DataFrame:
    """Retrieves Fama-French 5-Factor daily data."""
    q = f"SELECT date, mktrf, smb, hml, rmw, cma, rf FROM ff.fivefactors_daily WHERE date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'ff_date'})
    except Exception:
        return pd.DataFrame(columns=['ff_date', 'mktrf', 'smb', 'hml', 'rmw', 'cma', 'rf'])

def g_esg(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves ESG/Governance proxies dynamically."""
    q = f"SELECT meeting_date as as_of_date, female_directors, minority_directors FROM iss_directors_global.company_diversity WHERE ticker = '{t}' ORDER BY meeting_date ASC"
    try:
        return d.raw_sql(q, date_cols=['as_of_date'])
    except Exception:
        return pd.DataFrame(columns=['as_of_date', 'female_directors', 'minority_directors'])

def g_bdx(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves BoardEx Director Network data."""
    q = f"SELECT a.annual_report_date, AVG(b.network_size) as avg_board_network FROM boardex_row.row_wrds_org_summary a JOIN boardex_row.row_wrds_individual_networks b ON a.companyid = b.companyid WHERE a.ticker = '{t}' GROUP BY a.annual_report_date ORDER BY a.annual_report_date ASC"
    try:
        return d.raw_sql(q, date_cols=['annual_report_date'])
    except Exception:
        return pd.DataFrame(columns=['annual_report_date', 'avg_board_network'])

def g_shv(d: wrds.Connection, t: str, s: str, e: str) -> pd.DataFrame:
    """Retrieves short volume and interest data."""
    q = f"SELECT date, shortint FROM comp.sec_shortint WHERE tic = '{t}' AND date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'shv_date'})
    except Exception:
        return pd.DataFrame(columns=['shv_date', 'shortint'])

def g_bta(d: wrds.Connection, p: int, s: str, e: str) -> pd.DataFrame:
    """Retrieves WRDS Beta Suite daily data."""
    q = f"SELECT date, beta FROM betasuite.beta_daily WHERE permno = {p} AND date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'bta_date'})
    except Exception:
        return pd.DataFrame(columns=['bta_date', 'beta'])

def g_fac(d: wrds.Connection, s: str, e: str) -> pd.DataFrame:
    """Retrieves WRDS daily factors."""
    q = f"SELECT date, mkt, smb, hml, umd FROM wrdsapps.factors_daily WHERE date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'fac_date'})
    except Exception:
        return pd.DataFrame(columns=['fac_date', 'mkt', 'smb', 'hml', 'umd'])

def b_pipe(c: pd.DataFrame, i: pd.DataFrame, r: pd.DataFrame, e: pd.DataFrame, ff: pd.DataFrame, esg: pd.DataFrame, bdx: pd.DataFrame, shv: pd.DataFrame, bta: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    """Builds an aligned, forward-filled feature matrix mapping off-hours events to trading sessions.

    Args:
        c: Compustat fundamentals DataFrame.
        i: IBES estimates DataFrame.
        r: CRSP daily returns DataFrame.
        e: Capital IQ events DataFrame.
        ff: Fama-French factors DataFrame.
        esg: ISS ESG DataFrame.
        bdx: BoardEx DataFrame.
        shv: Short Volume DataFrame.
        bta: Beta Suite DataFrame.
        fac: Factors DataFrame.

    Returns:
        pd.DataFrame: Integrated and aligned feature matrix.
    """
    def _fmt(df: pd.DataFrame, col: str) -> pd.DataFrame:
        if not df.empty:
            df[col] = pd.to_datetime(df[col], errors='coerce').dt.normalize()
            return df.dropna(subset=[col]).sort_values(col)
        return df

    r = _fmt(r, 'dlycaldt')
    c = _fmt(c, 'datadate')
    i = _fmt(i, 'statpers')
    e = _fmt(e, 'announcedate')
    ff = _fmt(ff, 'ff_date')
    esg = _fmt(esg, 'as_of_date')
    bdx = _fmt(bdx, 'annual_report_date')
    shv = _fmt(shv, 'shv_date')
    bta = _fmt(bta, 'bta_date')
    fac = _fmt(fac, 'fac_date')

    if not i.empty:
        i['fpi'] = pd.to_numeric(i['fpi'], errors='coerce').astype('Int8')
        i = i.dropna(subset=['fpi'])
        i_pvt = i.groupby(['statpers', 'fpi'])[['meanest', 'medest', 'numest', 'stdev', 'highest', 'lowest', 'fpedats', 'curcode']].last().unstack()
        i_pvt.columns = [f"{col[0]}_fy{col[1]}" for col in i_pvt.columns]
        i = i_pvt.reset_index()

    m = pd.merge_asof(r, c, left_on='dlycaldt', right_on='datadate', direction='backward')

    if not i.empty: m = pd.merge_asof(m, i, left_on='dlycaldt', right_on='statpers', direction='backward')
    if not ff.empty: m = pd.merge_asof(m, ff, left_on='dlycaldt', right_on='ff_date', direction='backward')
    if not esg.empty: m = pd.merge_asof(m, esg, left_on='dlycaldt', right_on='as_of_date', direction='backward')
    if not bdx.empty: m = pd.merge_asof(m, bdx, left_on='dlycaldt', right_on='annual_report_date', direction='backward')
    if not shv.empty: m = pd.merge_asof(m, shv, left_on='dlycaldt', right_on='shv_date', direction='backward')
    if not bta.empty: m = pd.merge_asof(m, bta, left_on='dlycaldt', right_on='bta_date', direction='backward')
    if not fac.empty: m = pd.merge_asof(m, fac, left_on='dlycaldt', right_on='fac_date', direction='backward')

    if not e.empty:
        e_g = e.groupby('announcedate').agg({'keydeveventtypeid': list, 'headline': list}).reset_index().sort_values('announcedate')
        e_m = pd.merge_asof(e_g, r[['dlycaldt']], left_on='announcedate', right_on='dlycaldt', direction='forward')
        e_f = e_m.dropna(subset=['dlycaldt']).groupby('dlycaldt').agg({'keydeveventtypeid': 'sum', 'headline': 'sum'}).reset_index()
        m = pd.merge(m, e_f, on='dlycaldt', how='left')

    d_cols = ['datadate', 'statpers', 'ff_date', 'as_of_date', 'annual_report_date', 'shv_date', 'bta_date', 'fac_date']
    m.drop(columns=[col for col in d_cols if col in m.columns], inplace=True)

    return m

In [2]:
def g_sch(d: wrds.Connection, k: str) -> pd.DataFrame:
    """Lists schema.table pairs whose schema name matches a keyword."""
    q = "SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema LIKE %(k)s ORDER BY table_schema, table_name"
    return d.raw_sql(q, params={'k': f'%{k}%'})

def g_col(d: wrds.Connection, schema: str, table: str) -> pd.DataFrame:
    """Lists columns and their data types for a given schema.table."""
    q = "SELECT column_name, data_type FROM information_schema.columns WHERE table_schema = %(s)s AND table_name = %(t)s ORDER BY ordinal_position"
    return d.raw_sql(q, params={'s': schema, 't': table})

In [3]:
def g_sch(d: wrds.Connection, k: str) -> pd.DataFrame:
    """Lists schema.table pairs whose schema name matches a keyword."""
    q = "SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema LIKE %(k)s ORDER BY table_schema, table_name"
    return d.raw_sql(q, params={'k': f'%{k}%'})

def g_col(d: wrds.Connection, schema: str, table: str) -> pd.DataFrame:
    """Lists columns and their data types for a given schema.table."""
    q = "SELECT column_name, data_type FROM information_schema.columns WHERE table_schema = %(s)s AND table_name = %(t)s ORDER BY ordinal_position"
    return d.raw_sql(q, params={'s': schema, 't': table})

In [4]:
# def resolve_ids(d: wrds.Connection, name: str) -> Dict[str, pd.DataFrame]:
#     """Resolve a company's identifiers across WRDS databases by name.

#     Queries Compustat (Global + North America), IBES, CRSP, and Capital IQ
#     for any entity whose name matches the given pattern. Intended as a
#     one-shot diagnostic to run before the data loaders, so silent
#     empty-dataframe failures (caused by wrong gvkey/ticker/companyid) are
#     caught at resolution time rather than downstream.

#     Each source is defined by a (schema, table, name_col, select_cols) tuple.
#     Column names are validated against information_schema at call time, so
#     the function tolerates WRDS renaming columns between releases: any
#     requested column that doesn't exist is silently dropped from the SELECT.

#     Args:
#         d: Active WRDS connection.
#         name: Case-insensitive substring to match against company names.

#     Returns:
#         Dict keyed by source, each mapping to a DataFrame of candidate
#         identifiers. Empty DataFrames indicate no match in that source.
#     """
#     sources = {
#         'comp_g':  ('comp', 'g_company',     'conm',
#                     ['gvkey', 'conm', 'tic', 'exchg', 'fic', 'loc', 'costat']),
#         'comp_na': ('comp', 'company',       'conm',
#                     ['gvkey', 'conm', 'tic', 'exchg', 'fic', 'loc', 'costat']),
#         'ibes':    ('ibes', 'idsum',         'cname',
#                     ['ticker', 'cusip', 'cname', 'usfirm']),
#         'crsp':    ('crsp', 'stocknames_v2', 'issuernm',
#                     ['permno', 'permco', 'issuernm', 'ticker', 'primaryexch', 'hdrcusip']),
#         'ciq':     ('ciq',  'ciqcompany',    'companyname',
#                     ['companyid', 'companyname', 'city', 'simpleindustrydescription']),
#     }

#     col_q = ("SELECT column_name FROM information_schema.columns "
#              "WHERE table_schema = %(s)s AND table_name = %(t)s")
#     out: Dict[str, pd.DataFrame] = {}

#     for src, (schema, table, name_col, wanted) in sources.items():
#         try:
#             avail = set(d.raw_sql(col_q, params={'s': schema, 't': table})['column_name'])
#             if name_col not in avail:
#                 print(f"[{src:<7}] skipped: name column '{name_col}' missing in {schema}.{table}")
#                 out[src] = pd.DataFrame()
#                 continue
#             cols = [c for c in wanted if c in avail]
#             q = (f"SELECT DISTINCT {', '.join(cols)} FROM {schema}.{table} "
#                  f"WHERE {name_col} ILIKE %(n)s ORDER BY {cols[0]} LIMIT 50")
#             df = d.raw_sql(q, params={'n': f'%{name}%'})
#         except Exception as exc:
#             print(f"[{src:<7}] query failed: {exc}")
#             df = pd.DataFrame()

#         out[src] = df
#         n = len(df)
#         print(f"[{src:<7}] {n} match{'' if n == 1 else 'es'}")
#         if n:
#             print(df.to_string(index=False))

#     return out


# ids = resolve_ids(db, "BHP GROUP")

In [5]:
usr      = "zackienzle1"
gvkey    = "013312"
permno   = 75039
ibes_tic = "@BHP"
ciq_id   = "256654"
us_tic   = "BHP"
st       = "2000-01-01"
ed       = "2026-01-07"

db = wrds.Connection(wrds_username=usr)

Loading library list...
Done


In [6]:
# for kw in ['comp', 'ibes', 'crsp', 'ciq', 'ff', 'iss', 'boardex', 'betasuite', 'wrdsapps']:
#     print(kw)
#     print(g_sch(db, kw).to_string())

# targets = [
#     ('comp',      'g_security'),
#     ('comp',      'g_funda'),
#     ('comp',      'security'),
#     ('comp',      'funda'),
#     ('comp',      'sec_shortint'),
#     ('ibes',      'idsum'),
#     ('ibes',      'statsumu_epsus'),
#     ('crsp',      'dsf_v2'),
#     ('ciq',       'wrds_keydev'),
#     ('ciq',       'ciqkeydev'),
#     ('ciq',       'wrds_ticker'),
#     ('ff',        'fivefactors_daily'),
#     ('betasuite', 'beta_daily'),
#     ('wrdsapps',  'factors_daily'),
# ]
# for s, t in targets:
#     print(f"{s}.{t}")
#     print(g_col(db, s, t).to_string())

# print("BHP gvkey")
# print(db.raw_sql("SELECT gvkey, tic, dldtei, sedol, isin FROM comp.g_security WHERE tic = %(t)s", params={'t': 'BHP'}))
# print("BHP ibes")
# print(db.raw_sql("SELECT ticker, cusip, cname, sdates FROM ibes.idsum WHERE ticker = %(t)s ORDER BY sdates DESC LIMIT 5", params={'t': 'BHP'}))

# db.close()

In [7]:
df_cmp = g_cmp(db, gvkey)
df_ibs = g_ibs_int(db, ibes_tic)
df_crs = g_crs(db, permno, st, ed)
df_evt = g_evt(db, ciq_id)
df_ff = g_ff(db, st, ed)
df_esg = g_esg(db, ibes_tic)
df_bdx = g_bdx(db, ibes_tic)
df_shv = g_shv(db, us_tic, st, ed)
df_bta = g_bta(db, permno, st, ed)
df_fac = g_fac(db, st, ed)

db.close()

df_main = b_pipe(df_cmp, df_ibs, df_crs, df_evt, df_ff, df_esg, df_bdx, df_shv, df_bta, df_fac)

In [8]:
df_main

,dlycaldt,dlyret,dlyvol,curcd,at,lt,ni,revt,meanest_fy1,meanest_fy2,...,curcode_fy1,curcode_fy2,mktrf,smb,hml,rmw,cma,rf,keydeveventtypeid,headline
0,2000-01-03,-0.025882,27900.0,AUD,31487.0,22126.0,-2312.0,19229.0,0.381,0.446,...,AUD,AUD,-0.0071,-0.0009,-0.0131,-0.0148,-0.007,0.0002,"[80, 80, 80, 80, 81, 81, 81, 81, 95, 80, 81, 9...",[Newmont Australia Limited signed a merger agr...
1,2000-01-04,0.036232,45800.0,AUD,31487.0,22126.0,-2312.0,19229.0,0.381,0.446,...,AUD,AUD,-0.0406,0.0034,0.0207,0.0053,0.0136,0.0002,NaN,NaN
2,2000-01-05,0.011655,96100.0,AUD,31487.0,22126.0,-2312.0,19229.0,0.381,0.446,...,AUD,AUD,-0.0009,0.0036,-0.0005,0.0045,0.0115,0.0002,NaN,NaN
3,2000-01-06,-0.016129,19800.0,AUD,31487.0,22126.0,-2312.0,19229.0,0.381,0.446,...,AUD,AUD,-0.0074,-0.0004,0.0124,0.0064,0.0121,0.0002,NaN,NaN
4,2000-01-07,0.067916,119400.0,AUD,31487.0,22126.0,-2312.0,19229.0,0.381,0.446,...,AUD,AUD,0.0321,-0.0089,-0.0157,-0.0083,-0.01,0.0002,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6534,2025-12-24,-0.003438,854986.0,USD,108790.0,56572.0,9019.0,51262.0,2.19,2.03,...,USD,USD,0.0029,0.0007,0.0001,-0.0005,0.0023,0.0002,NaN,NaN
6535,2025-12-26,0.016757,1436228.0,USD,108790.0,56572.0,9019.0,51262.0,2.19,2.03,...,USD,USD,-0.0006,-0.0022,0.0009,0.0057,0.0024,0.0002,NaN,NaN
6536,2025-12-29,-0.024237,2539524.0,USD,108790.0,56572.0,9019.0,51262.0,2.19,2.03,...,USD,USD,-0.0041,-0.0017,0.0007,0.0032,0.0002,0.0002,NaN,NaN
6537,2025-12-30,0.008776,2178951.0,USD,108790.0,56572.0,9019.0,51262.0,2.19,2.03,...,USD,USD,-0.002,-0.0049,0.0028,0.0036,0.0013,0.0002,NaN,NaN


In [9]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6539 entries, 0 to 6538
Data columns (total 32 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   dlycaldt           6539 non-null   datetime64[ns]
 1   dlyret             6539 non-null   Float64       
 2   dlyvol             6539 non-null   Float64       
 3   curcd              6539 non-null   string        
 4   at                 6539 non-null   Float64       
 5   lt                 6539 non-null   Float64       
 6   ni                 6539 non-null   Float64       
 7   revt               6539 non-null   Float64       
 8   meanest_fy1        6539 non-null   Float64       
 9   meanest_fy2        6539 non-null   Float64       
 10  medest_fy1         6539 non-null   Float64       
 11  medest_fy2         6539 non-null   Float64       
 12  numest_fy1         6539 non-null   Float64       
 13  numest_fy2         6539 non-null   Float64       
 14  stdev_fy

In [10]:
def s_csv(d_m: Dict[str, pd.DataFrame], d_p: str = "../data") -> None:
    """Saves multiple DataFrames to CSV efficiently."""
    p = Path(d_p)
    p.mkdir(parents=True, exist_ok=True)
    for k, v in d_m.items():
        if not v.empty:
            v.to_csv(p / f"{k}.csv", index=False, chunksize=100000)

In [11]:
d_out = {
    "bhp_cmp": df_cmp,
    "bhp_ibs": df_ibs,
    "bhp_crs": df_crs,
    "bhp_evt": df_evt,
    "bhp_ff": df_ff,
    "bhp_shv": df_shv,
    "bhp_main": df_main
}

s_csv(d_out)
print(f"Pipeline executed. Main matrix shape: {df_main.shape}")

Pipeline executed. Main matrix shape: (6539, 32)
